# События

## Клиенты

In [ ]:
{
  "event_id": "evt_987654321",
  "event_type": "client_profile_updated",
  "timestamp": "2026-09-10T14:30:00Z",
  "payload": {
    "client_id": "cl_10045",
    "full_name": "Konstantin Amelin",
    "gender": "Male",
    "birthday": "1999-05-27",
    "salutation": "Mr.",
    "preferred_payment_method": "cash",
    "phone": "+66123456789",
    "email": "const123@gmail.com",
    "nationality": "Russian",
    "residential_address": "Sukhumvit 71, Bangkok",
    "driving_experience_years": 5, # Представим как будто это будет рассчитано в самом приложении на основе фотографии водительских прав
    "drive_license_category": "local", # Здесь либо метсные либо международные права
    "drive_license_photo": "s3://topbikes-docs/licenses/cl_10045.jpg",
    "is_first_order": True,
    "rating_score": 5.0 # Будет рассчитываться внутри приложения в зависимости от количества заказов, опыта и т.п.
  }
}

## Бронирование аренды клиентом

In [ ]:
{
  "event_id": "evt_987654322",
  "event_type": "rental_booked",
  "timestamp": "2026-09-12T09:15:00Z",
  "payload": {
    "rental_id": "rnt_5001",
    "client_id": "cl_10045",
    "bike_id": "bk_adv160_04",
    "location_id": "loc_bkk_phra_khanong",
    "planned_start_date": "2026-09-15",
    "planned_end_date": "2026-09-29",
    "rental_price_daily": 15.00,
    "deposit_amount": 200.00,
    "discount_amount": 10.00 # Разовый промокод на скидку
  }
}

## Выдача байка

In [ ]:
{
  "event_id": "evt_mech_001",
  "event_type": "bike_dispensed",
  "timestamp": "2026-09-15T10:00:00Z",
  "payload": {
    "rental_id": "rnt_5001",
    "mechanic_id": "mech_08",
    "location_id": "loc_bkk_phra_khanong",
    "actual_start_mileage": 12540,
    "fuel_level_percent": 100,
    "pre_rental_condition_score": 9,
    "helmets_issued_qty": 2
  }
}

## Возврат байка

In [ ]:
{
  "event_id": "evt_mech_002",
  "event_type": "bike_returned",
  "timestamp": "2026-09-29T11:30:00Z",
  "payload": {
    "rental_id": "rnt_5001",
    "mechanic_id": "mech_08",
    "location_id": "loc_bkk_phra_khanong", 
    "actual_end_mileage": 13120,
    "fuel_level_percent": 20,
    "post_rental_condition_score": 8,
    "deposit_deducted_amount": 45.00,
    "deduction_reason_code": "damage_scratch"
  }
}

## Полевое обслуживание

In [ ]:
{
  "event_id": "evt_987654324",
  "event_type": "field_service_performed",
  "timestamp": "2026-09-22T16:45:00Z",
  "payload": {
    "rental_id": "rnt_5001",
    "mechanic_id": "mech_02",
    "issue_category": "dead_battery",
    "resolution_action": "battery_replacement",
    "service_cost_usd": 45.00,
    "client_fault_flag": True
  }
}

# Скрипт для генерации данных

### Dim_bikes

In [2]:
import pandas as pd
import random
import string

# Фиксируем seed для воспроизводимости
random.seed(42)

# Справочник уникальных моделей с реальными ТТХ и средними ценами закупки (в THB)
MODELS_CATALOG = [
    {
        "brand": "Honda",
        "title": "Honda Click 125i",
        "type": "city bike",
        "hourse_power": 11.0,
        "engine_volume": 125,
        "gearbox_type": "automatic",
        "usb_charging_availability": False,
        "procurement_price": 54000,
        "colors": ["Matte Black", "White-Red", "Blue"],
        "count": 8,
    },
    {
        "brand": "Honda",
        "title": "Honda Wave 110i",
        "type": "city bike",
        "hourse_power": 8.8,
        "engine_volume": 109,
        "gearbox_type": "semi-automatic",
        "usb_charging_availability": False,
        "procurement_price": 46000,
        "colors": ["Red-Black", "Blue", "Black"],
        "count": 6,
    },
    {
        "brand": "Honda",
        "title": "Honda ADV 160",
        "type": "city bike",
        "hourse_power": 15.8,
        "engine_volume": 157,
        "gearbox_type": "automatic",
        "usb_charging_availability": True,
        "procurement_price": 99000,
        "colors": ["Matte Red", "Matte Black", "Grey", "Pearl Blue"],
        "count": 8,
    },
    {
        "brand": "Yamaha",
        "title": "Yamaha NMAX 155",
        "type": "city bike",
        "hourse_power": 15.1,
        "engine_volume": 155,
        "gearbox_type": "automatic",
        "usb_charging_availability": True,
        "procurement_price": 93000,
        "colors": ["Dark Petrol", "Silver", "Matte Black"],
        "count": 7,
    },
    {
        "brand": "Honda",
        "title": "Honda Forza 350",
        "type": "touring",
        "hourse_power": 29.2,
        "engine_volume": 330,
        "gearbox_type": "automatic",
        "usb_charging_availability": True,
        "procurement_price": 179000,
        "colors": ["Matte Black", "Smoky Grey", "Red"],
        "count": 5,
    },
    {
        "brand": "Yamaha",
        "title": "Yamaha XMAX 300",
        "type": "touring",
        "hourse_power": 27.6,
        "engine_volume": 292,
        "gearbox_type": "automatic",
        "usb_charging_availability": True,
        "procurement_price": 189000,
        "colors": ["Icon Blue", "Dark Magma", "Sonic Grey"],
        "count": 5,
    },
    {
        "brand": "Honda",
        "title": "Honda CB500X",
        "type": "touring",
        "hourse_power": 47.0,
        "engine_volume": 471,
        "gearbox_type": "manual",
        "usb_charging_availability": False,
        "procurement_price": 224000,
        "colors": ["Grand Prix Red", "Matte Gunpowder Black", "Pearl Organic Green"],
        "count": 4,
    },
    {
        "brand": "Honda",
        "title": "Honda CRF300L",
        "type": "offroad",
        "hourse_power": 27.0,
        "engine_volume": 286,
        "gearbox_type": "manual",
        "usb_charging_availability": False,
        "procurement_price": 145000,
        "colors": ["Extreme Red", "Grey"],
        "count": 4,
    },
    {
        "brand": "Kawasaki",
        "title": "Kawasaki KLX230",
        "type": "offroad",
        "hourse_power": 19.0,
        "engine_volume": 233,
        "gearbox_type": "manual",
        "usb_charging_availability": False,
        "procurement_price": 125000,
        "colors": ["Lime Green", "Battle Gray"],
        "count": 4,
    },
    {
        "brand": "Royal Enfield",
        "title": "Royal Enfield Classic 350",
        "type": "city bike",
        "hourse_power": 20.2,
        "engine_volume": 349,
        "gearbox_type": "manual",
        "usb_charging_availability": True,
        "procurement_price": 139000,
        "colors": ["Halcyon Black", "Signals Desert Sand", "Bronze"],
        "count": 4,
    },
]

def generate_mock_vin(brand: str) -> str:
    # WMI префиксы для реализма: MLH = Honda (Thailand), ML3 = Yamaha (Thailand), ME3 = Royal Enfield
    wmi_map = {"Honda": "MLH", "Yamaha": "ML3", "Kawasaki": "MLK", "Royal Enfield": "ME3"}
    wmi = wmi_map.get(brand, "MLH")
    chars = "".join(random.choices(string.ascii_uppercase + string.digits, k=14))
    return f"{wmi}{chars}"

def generate_plate_number() -> str:
    digit = random.randint(1, 9)
    thai_char_pairs = ["กข", "ขค", "งจ", "ฉช", "ฎฏ", "ทธ", "พภ", "มย", "รล", "วศ"]
    letters = random.choice(thai_char_pairs)
    num = random.randint(100, 9999)
    return f"{digit}{letters}-{num}"

# Формируем итоговые записи
bikes = []
bike_id_counter = 1001

for model in MODELS_CATALOG:
    for _ in range(model["count"]):
        bikes.append({
            "bike_id": bike_id_counter,
            "WIN_number": generate_mock_vin(model["brand"]),
            "plate_number": generate_plate_number(),
            "title": model["title"],
            "type": model["type"],
            "model_year": random.choice([2023, 2024, 2024, 2025, 2025, 2026]),
            "color": random.choice(model["colors"]),
            "brand": model["brand"],
            "hourse_power": model["hourse_power"],
            "engine_volume (cc)": model["engine_volume"],
            "gearbox_type": model["gearbox_type"],
            "usb_charging_availability": model["usb_charging_availability"],
            "procurement_price": model["procurement_price"]
        })
        bike_id_counter += 1

df_bikes = pd.DataFrame(bikes)

In [27]:
df_bikes.head()

,bike_id,WIN_number,plate_number,title,type,model_year,color,brand,hourse_power,engine_volume (cc),gearbox_type,usb_charging_availability,procurement_price
0,1001,MLHXAJI0Y6DPBHSAH,9พภ-3711,Honda Click 125i,city bike,2025,Blue,Honda,11.0,125,automatic,False,54000
1,1002,MLHK51FPKH1DNMMJB,8รล-2145,Honda Click 125i,city bike,2025,Matte Black,Honda,11.0,125,automatic,False,54000
2,1003,MLHT3W5UZBIKCIDKW,6งจ-6165,Honda Click 125i,city bike,2024,Matte Black,Honda,11.0,125,automatic,False,54000
3,1004,MLHYZYCWTIQJ7YHL1,1ฉช-625,Honda Click 125i,city bike,2024,White-Red,Honda,11.0,125,automatic,False,54000
4,1005,MLHJH75LXO6QJIUJV,7ทธ-3693,Honda Click 125i,city bike,2024,Blue,Honda,11.0,125,automatic,False,54000


ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/Users/konstantinamelin/Library/Python/3.14/lib/python/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.result()
    ~~~~~~~~^^
  File "/Users/konstantinamelin/Library/Python/3.14/lib/python/site-packages/ipykernel/kernelbase.py", line 341, in dispatch_control
    await self.process_control(msg)
  File "/Users/konstantinamelin/Library/Python/3.14/lib/python/site-packages/ipykernel/kernelbase.py", line 347, in process_control
    idents, msg = self.session.feed_identities(msg, copy=False)
                  ~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/Users/konstantinamelin/Library/Python/3.14/lib/python/site-packages/jupyter_client/session.py", line 994, in feed_identities
    raise ValueError(msg)
ValueError: DELIM not in msg_list
ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/Users/konsta

### Генерация событий

In [20]:
import time
import json
import random
from datetime import datetime, timezone
import pandas as pd
from faker import Faker

try:
    from kafka import KafkaProducer
except ImportError:
    KafkaProducer = None

fake = Faker()

def run_generator(mode="pandas", duration_seconds=60):
    print(f"Инициализация генератора (Режим: {mode.upper()})")
    
    # 1. Загрузка справочника байков из CSV
    try:
        df_bikes = pd.read_csv('df_bikes.csv')
        BIKES = df_bikes['bike_id'].tolist()
        print(f"✅ Успешно загружен справочник байков: {len(BIKES)} шт.")
    except Exception as e:
        print(f"❌ Ошибка загрузки df_bikes.csv. Убедитесь, что файл лежит в корне проекта. Ошибка: {e}")
        return

    # 2. Настройка получателя данных (Kafka или словари для Pandas)
    producer = None
    if mode == "kafka":
        if KafkaProducer is None:
            print("❌ Библиотека kafka-python не установлена.")
            return
        try:
            producer = KafkaProducer(
                bootstrap_servers=['localhost:9092'],
                value_serializer=lambda v: json.dumps(v).encode('utf-8')
            )
            print("✅ Успешно подключено к Redpanda (Kafka)")
        except Exception as e:
            print(f"❌ Ошибка подключения к Kafka: {e}")
            return
            
    # Внутреннее хранилище для Pandas
    pandas_store = {
        'client_profile_updated': [],
        'rental_booked': [],
        'bike_dispensed': [],
        'field_service_performed': [],
        'bike_returned': []
    }

    # 3. Состояние (State Machine)
    state = {
        "clients": [],
        "pending_rentals": [],
        "active_rentals": []
    }

    LOCATIONS = ["loc_bkk_phra_khanong", "loc_bkk_bang_na", "loc_bkk_asoke", "loc_bkk_sathorn"]
    MECHANICS = ["mech_01", "mech_02", "mech_03", "mech_04"]

    # 4. Функция маршрутизации событий (в Kafka или в Pandas)
    def send_event(topic, event_type, payload):
        timestamp = datetime.now(timezone.utc).isoformat()
        event_id = f"evt_{fake.uuid4()}"
        
        if mode == "kafka":
            event = {
                "event_id": event_id,
                "event_type": event_type,
                "timestamp": timestamp,
                "payload": payload
            }
            producer.send(topic, event)
            producer.flush()
            print(f"[{topic}] {event_type} -> {payload.get('client_id') or payload.get('rental_id')}")
        
        elif mode == "pandas":
            # Разворачиваем payload в плоский словарь для красивого датафрейма
            row = {
                "event_id": event_id,
                "event_type": event_type,
                "timestamp": timestamp
            }
            row.update(payload) # Добавляем все ключи из payload как отдельные колонки
            pandas_store[event_type].append(row)
            print(f"[Pandas] Сгенерировано событие: {event_type}")

    # 5. Логика генерации бизнес-событий
    def generate_client():
        client_id = f"cl_{fake.unique.random_int(min=10000, max=99999)}"
        state["clients"].append(client_id)
        payload = {
            "client_id": client_id,
            "full_name": fake.name(),
            "gender": random.choice(["Male", "Female"]),
            "birthday": fake.date_of_birth(minimum_age=18, maximum_age=60).isoformat(),
            "phone": fake.phone_number(),
            "email": fake.email(),
            "nationality": random.choice(["Russian", "Thai", "British", "American", "German", "French"]),
            "residential_address": fake.address().replace('\n', ', '),
            "preferred_payment_method": random.choice(["cash", "credit_card", "promptpay", "alipay"]),
            "driving_experience_years": random.randint(1, 15),
            "drive_license_category": random.choice(["local", "international"]),
            "drive_license_verified": True,
            "rating_score": round(random.uniform(3.5, 5.0), 1)
        }
        send_event('client_app_events', 'client_profile_updated', payload)

    def generate_booking():
        if not state["clients"]: return
        rental_id = f"rnt_{fake.unique.random_int(min=5000, max=99999)}"
        state["pending_rentals"].append(rental_id)
        payload = {
            "rental_id": rental_id,
            "client_id": random.choice(state["clients"]),
            "bike_id": random.choice(BIKES), # Берем реальный ID из датасета!
            "location_id": random.choice(LOCATIONS),
            "rental_price_daily": random.choice([15.00, 20.00, 25.00]),
            "deposit_amount": 200.00,
            "discount_amount": random.choice([0.00, 0.00, 5.00, 10.00])
        }
        send_event('client_app_events', 'rental_booked', payload)

    def generate_dispense():
        if not state["pending_rentals"]: return
        rental_id = state["pending_rentals"].pop(0)
        state["active_rentals"].append(rental_id)
        payload = {
            "rental_id": rental_id,
            "mechanic_id": random.choice(MECHANICS),
            "location_id": random.choice(LOCATIONS),
            "actual_start_mileage": random.randint(1000, 20000),
            "fuel_level_percent": random.choice([50, 75, 100]),
            "pre_rental_condition_score": random.randint(7, 10),
            "helmets_issued_qty": random.randint(1, 2),
            "actual_payment_method": random.choice(["cash", "credit_card", "promptpay"])
        }
        send_event('mechanic_app_events', 'bike_dispensed', payload)

    def generate_field_service():
        if not state["active_rentals"]: return
        rental_id = random.choice(state["active_rentals"])
        payload = {
            "rental_id": rental_id,
            "mechanic_id": random.choice(MECHANICS),
            "issue_category": random.choice(["dead_battery", "flat_tire", "engine_check", "lost_key"]),
            "resolution_action": "fixed_on_site",
            "service_cost_usd": round(random.uniform(20.0, 100.0), 2),
            "client_fault_flag": random.choice([True, False])
        }
        send_event('mechanic_app_events', 'field_service_performed', payload)

    def generate_return():
        if not state["active_rentals"]: return
        rental_id = state["active_rentals"].pop(0)
        payload = {
            "rental_id": rental_id,
            "mechanic_id": random.choice(MECHANICS),
            "location_id": random.choice(LOCATIONS),
            "actual_end_mileage": random.randint(20100, 25000),
            "fuel_level_percent": random.choice([10, 20, 50, 100]),
            "post_rental_condition_score": random.randint(5, 10),
            "deposit_deducted_amount": random.choice([0.00, 0.00, 45.00, 100.00]),
            "deduction_reason_code": "damage_scratch"
        }
        send_event('mechanic_app_events', 'bike_returned', payload)

    # 6. Главный цикл генерации
    start_time = time.time()
    
    if mode == "pandas":
        print(f"Сбор данных запущен. Скрипт автоматически остановится через {duration_seconds} секунд...")
    else:
        print("Данные отправляются в Kafka. Нажмите Ctrl+C для остановки.")
        
    try:
        while True:
            # Остановка по таймеру для режима Pandas
            if mode == "pandas" and (time.time() - start_time) > duration_seconds:
                break
                
            action = random.choices(
                [generate_client, generate_booking, generate_dispense, generate_field_service, generate_return],
                weights=[25, 30, 20, 5, 20],
                k=1
            )[0]
            action()
            
            # В Pandas режиме спим совсем чуть-чуть, чтобы сгенерировать побольше данных
            sleep_time = random.uniform(0.05, 0.2) if mode == "pandas" else random.uniform(1.0, 4.0)
            time.sleep(sleep_time)
            
    except KeyboardInterrupt:
        print("\n🛑 Остановка по требованию пользователя.")
        
    # 7. Выгрузка в CSV (только для Pandas)
    if mode == "pandas":
        print("\n💾 Формирование CSV файлов...")
        for event_type, rows in pandas_store.items():
            if rows: # Если список не пустой
                df = pd.DataFrame(rows)
                filename = f"sample_{event_type}.csv"
                df.to_csv(filename, index=False, encoding='utf-8')
                print(f"✅ Сохранен файл: {filename} (строк: {len(df)})")
        print("🎉 Готово! Можете проверять CSV файлы.")

In [21]:
run_generator(mode="pandas", duration_seconds=60)
    
# Для отправки в Kafka:
# run_generator(mode="kafka")

Инициализация генератора (Режим: PANDAS)
✅ Успешно загружен справочник байков: 55 шт.
Сбор данных запущен. Скрипт автоматически остановится через 60 секунд...
[Pandas] Сгенерировано событие: client_profile_updated
[Pandas] Сгенерировано событие: rental_booked
[Pandas] Сгенерировано событие: rental_booked
[Pandas] Сгенерировано событие: rental_booked
[Pandas] Сгенерировано событие: client_profile_updated
[Pandas] Сгенерировано событие: rental_booked
[Pandas] Сгенерировано событие: rental_booked
[Pandas] Сгенерировано событие: bike_dispensed
[Pandas] Сгенерировано событие: bike_dispensed
[Pandas] Сгенерировано событие: rental_booked
[Pandas] Сгенерировано событие: bike_returned
[Pandas] Сгенерировано событие: field_service_performed
[Pandas] Сгенерировано событие: client_profile_updated
[Pandas] Сгенерировано событие: client_profile_updated
[Pandas] Сгенерировано событие: client_profile_updated
[Pandas] Сгенерировано событие: client_profile_updated
[Pandas] Сгенерировано событие: bike_re

In [22]:
bike_dispensed = pd.read_csv('sample_bike_dispensed.csv')
bike_returned = pd.read_csv('sample_bike_returned.csv')
client_profile_updated = pd.read_csv('sample_client_profile_updated.csv')
field_service_performed = pd.read_csv('sample_field_service_performed.csv')
rental_booked = pd.read_csv('sample_rental_booked.csv')

In [ ]:
df_bikes.head()

: 

In [ ]:
#bike_dispensed.head()
#bike_returned.head()
client_profile_updated.head()
#field_service_performed.head()
#rental_booked.head()

: 